# Multi-agent systems

The more you delegate task or longer the query or context for a single model, it degrade the performance. Instead we build Mult-agent systems for that type of problem, so that each agent will have their own role and task to do. It's like Single Responsibility Principle. In this way, agents or model perform really well.

How to achieve this ?
In Langchain its pretty easy, they are basically tools called by a supervisor or the main agent.

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

# Creating Subagents

In [5]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2


# create subagents

subagent_1 = create_agent(
    model="claude-sonnet-4-5",
    tools=[square_root]
)

subagent_2 = create_agent(
    model="claude-sonnet-4-5",
    tools=[square]
)

# Calling Subagents

In [9]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages":[HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content


# creating the main agent

main_agent = create_agent(
    model="claude-sonnet-4-5",
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number."
)


# Test

In [10]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [11]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='ad72c4b4-fc59-447e-a311-8bb1a66376fc'),
              AIMessage(content=[{'id': 'toolu_019Cfu7uoKfi9Te4xg5667Ac', 'caller': {'type': 'direct'}, 'input': {'x': 456}, 'name': 'call_subagent_1', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_016Zi4WP5nyZZ2Mc4xa36p64', 'container': None, 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 672, 'output_tokens': 57, 'server_tool_use': None, 'service_tier': 'standard'}, 'stop_details': None, 'model_name': 'claude-sonnet-4-5-20250929', 'model_provider': 'anthropic'}, id='lc_run--019e00a3-9ba2-7fd1-b717-ed1dff2d1ec9-0', tool_calls=[{'name': 'call_subagent_1', 'a